# 03. Limpieza y base analítica

Este notebook carga los módulos anuales, selecciona variables, valida llaves y construye la base analítica inicial.

In [3]:
from pathlib import Path
import sys

print("1. Imports básicos correctos")

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)

print("2. Directorio actual:", Path.cwd())
print("3. Raíz detectada:", PROJECT_ROOT)

ruta_src = str(PROJECT_ROOT / "src")

if ruta_src not in sys.path:
    sys.path.insert(0, ruta_src)

print("4. Ruta src:", ruta_src)
print("5. Existe src:", (PROJECT_ROOT / "src").exists())

1. Imports básicos correctos
2. Directorio actual: c:\Users\neo_a\OneDrive\Documents\Unicafam\Quinto Semestre\Analitica descriptiva II\Proyecto Integrador\notebooks
3. Raíz detectada: C:\Users\neo_a\OneDrive\Documents\Unicafam\Quinto Semestre\Analitica descriptiva II\Proyecto Integrador
4. Ruta src: C:\Users\neo_a\OneDrive\Documents\Unicafam\Quinto Semestre\Analitica descriptiva II\Proyecto Integrador\src
5. Existe src: True


In [4]:
import config

print("config importado correctamente")
print("Archivo usado:", config.__file__)
print("ANNUAL_MODULES:", config.ANNUAL_MODULES)
print("ANALYTIC_BASE:", config.ANALYTIC_BASE)

config importado correctamente
Archivo usado: C:\Users\neo_a\OneDrive\Documents\Unicafam\Quinto Semestre\Analitica descriptiva II\Proyecto Integrador\src\config.py
ANNUAL_MODULES: C:\Users\neo_a\OneDrive\Documents\Unicafam\Quinto Semestre\Analitica descriptiva II\Proyecto Integrador\data_processed\annual_modules
ANALYTIC_BASE: C:\Users\neo_a\OneDrive\Documents\Unicafam\Quinto Semestre\Analitica descriptiva II\Proyecto Integrador\data_processed\analytic_base


In [5]:
import utils

print("utils importado correctamente")
print("Archivo usado:", utils.__file__)

utils importado correctamente
Archivo usado: C:\Users\neo_a\OneDrive\Documents\Unicafam\Quinto Semestre\Analitica descriptiva II\Proyecto Integrador\src\utils.py


In [6]:
from config import ANNUAL_MODULES, ANALYTIC_BASE, KEY_COLUMNS
from utils import estandarizar_columnas, validar_llaves, resumen_faltantes, exportar_csv

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 300)

print("Todo importado correctamente")

Todo importado correctamente


In [7]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.append(str(PROJECT_ROOT / 'src'))

from config import ANNUAL_MODULES, ANALYTIC_BASE, KEY_COLUMNS
from utils import estandarizar_columnas, validar_llaves, resumen_faltantes, exportar_csv

pd.set_option('display.max_columns', 300)

In [8]:
car = pd.read_csv(ANNUAL_MODULES / 'geih_2025_caracteristicas.csv', low_memory=False)
ft = pd.read_csv(ANNUAL_MODULES / 'geih_2025_fuerza_trabajo.csv', low_memory=False)
oc = pd.read_csv(ANNUAL_MODULES / 'geih_2025_ocupados.csv', low_memory=False)

car = estandarizar_columnas(car)
ft = estandarizar_columnas(ft)
oc = estandarizar_columnas(oc)

car.shape, ft.shape, oc.shape

((817550, 57), (653839, 44), (295810, 39))

In [9]:
print('CAR', validar_llaves(car, KEY_COLUMNS + ['PERIODOMES']))
print('FT', validar_llaves(ft, KEY_COLUMNS + ['PERIODOMES']))
print('OC', validar_llaves(oc, KEY_COLUMNS + ['PERIODOMES']))

CAR {'llaves_disponibles': ['DIRECTORIO', 'ORDEN', 'HOGAR', 'REGIS'], 'duplicados': 0}
FT {'llaves_disponibles': ['DIRECTORIO', 'ORDEN', 'HOGAR', 'REGIS'], 'duplicados': 0}
OC {'llaves_disponibles': ['DIRECTORIO', 'ORDEN', 'HOGAR', 'REGIS'], 'duplicados': 0}


In [10]:
llaves = [c for c in ['PERIODOMES', 'DIRECTORIO', 'SECUENCIAP', 'ORDEN', 'HOGAR', 'REGIS'] if c in car.columns and c in ft.columns and c in oc.columns]
llaves

['DIRECTORIO', 'ORDEN', 'HOGAR', 'REGIS']

In [17]:
vars_car = [c for c in ['PERIODOMES', 'DIRECTORIO', 'SECUENCIAP', 'ORDEN', 'HOGAR', 'REGIS', 'P3271', 'P6040', 'P6210', 'AREA', 'CLASE', 'DPTO', 'FEXC18'] if c in car.columns]
vars_ft = [c for c in ['PERIODOMES', 'DIRECTORIO', 'SECUENCIAP', 'ORDEN', 'HOGAR', 'REGIS', 'FT', 'PET'] if c in ft.columns]
vars_oc = [c for c in ['PERIODOMES', 'DIRECTORIO', 'SECUENCIAP', 'ORDEN', 'HOGAR', 'REGIS'] if c in oc.columns]

car_s = car[vars_car].copy()
ft_s = ft[vars_ft].copy()
oc_s = oc[vars_oc].copy()

base = car_s.merge(ft_s, on=llaves, how='left').merge(oc_s, on=llaves, how='left')
base.shape

(817550, 11)

In [22]:
base['SEXO'] = base['P3271'] if 'P3271' in base.columns else np.nan
base['EDAD'] = pd.to_numeric(base['P6040'], errors='coerce') if 'P6040' in base.columns else np.nan

base['GRUPO_EDAD'] = pd.cut(base['EDAD'], bins=[14,24,34,44,54,64,120], labels=['15-24','25-34','35-44','45-54','55-64','65+'])

base.head()

,DIRECTORIO,ORDEN,HOGAR,REGIS,P3271,P6040,AREA,CLASE,DPTO,FT,PET,SEXO,EDAD,GRUPO_EDAD
0,8086872,1,1,10,2,54,NaN,2,13,NaN,NaN,2,54,45-54
1,8086874,1,1,10,2,79,NaN,2,13,NaN,NaN,2,79,65+
2,8086874,2,1,10,1,60,NaN,2,13,NaN,NaN,1,60,55-64
3,8086874,3,1,10,1,59,NaN,2,13,NaN,NaN,1,59,55-64
4,8086875,1,1,10,2,80,NaN,2,13,NaN,NaN,2,80,65+


In [23]:
resumen_faltantes(base).head(20)

,variable,n_faltantes,pct_faltantes
10,PET,817550,1.000000
9,FT,817550,1.000000
6,AREA,224252,0.274298
13,GRUPO_EDAD,163711,0.200246
0,DIRECTORIO,0,0.000000
1,ORDEN,0,0.000000
5,P6040,0,0.000000
4,P3271,0,0.000000
3,REGIS,0,0.000000
2,HOGAR,0,0.000000


In [24]:
salida = ANALYTIC_BASE / 'base_analitica_geih_2025.csv'
exportar_csv(base, salida)
print(salida)

C:\Users\neo_a\OneDrive\Documents\Unicafam\Quinto Semestre\Analitica descriptiva II\Proyecto Integrador\data_processed\analytic_base\base_analitica_geih_2025.csv
